In [1]:
# Task 1: In day13.ipynb, define the required fields for each file
from pathlib import Path

DATA_DIR = Path("../archive")
REQUIRED_FIELDS = {
    "olist_order_items_dataset.csv": ["order_id", "product_id", "price", "freight_value"],
    "olist_orders_dataset.csv": ["order_id", "customer_id", "order_status", "order_purchase_timestamp"],
    "olist_customers_dataset.csv": ["customer_id", "customer_zip_code_prefix", "customer_city"],
}
print(REQUIRED_FIELDS)

{'olist_order_items_dataset.csv': ['order_id', 'product_id', 'price', 'freight_value'], 'olist_orders_dataset.csv': ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp'], 'olist_customers_dataset.csv': ['customer_id', 'customer_zip_code_prefix', 'customer_city']}


In [2]:
# Task 2: Validate every row and collect errors
import csv


def read_rows(path):
    with open(path, newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        return reader.fieldnames, list(reader)


def validate_row(row, required):
    errors = []
    for field in required:
        if not row.get(field):
            errors.append("missing " + field)
    return errors


def validate_every_row(rows, required):
    results = []
    for row in rows:
        errors = validate_row(row, required)
        errors += check_types(row)
        errors += check_business_rules(row)
        errors += check_regex(row)
        results.append((row, errors))
    return results


print("Validation functions are ready")

Validation functions are ready


In [3]:
# Task 3: Write rejected rows to bad_records.csv with a reason
def write_bad_records(bad_rows):
    with open("bad_records.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["row", "reason"])
        for row, errors in bad_rows:
            writer.writerow([row, "; ".join(errors)])
    print("Wrote bad_records.csv")

In [4]:
# Task 4: Check data types (numeric price, parseable dates)
from datetime import datetime


def check_types(row):
    errors = []
    if row.get("price"):
        try:
            float(row["price"])
        except ValueError:
            errors.append("price is not a number")
    for field in ["shipping_limit_date", "order_purchase_timestamp"]:
        if row.get(field):
            try:
                datetime.strptime(row[field], "%Y-%m-%d %H:%M:%S")
            except ValueError:
                errors.append(field + " is not a date")
    return errors


print("Type checks are ready")

Type checks are ready


In [5]:
# Task 5: Business rules: price > 0, order_status in the allowed list
ALLOWED_STATUS = ["delivered", "shipped", "canceled", "unavailable", "invoiced", "processing", "created", "approved"]


def check_business_rules(row):
    errors = []
    if row.get("price") and float(row["price"]) <= 0:
        errors.append("price must be positive")
    if row.get("order_status") and row["order_status"] not in ALLOWED_STATUS:
        errors.append("bad order status")
    return errors


print("Business rules are ready")

Business rules are ready


In [6]:
# Task 6: Regex: zip prefix must be 5 digits; IDs must be 32 hex characters
import re


def check_regex(row):
    errors = []
    for field in ["customer_zip_code_prefix", "seller_zip_code_prefix"]:
        if row.get(field) and not re.match(r"^\d{5}$", row[field]):
            errors.append(field + " must be 5 digits")
    for field in ["order_id", "customer_id", "product_id", "seller_id"]:
        if row.get(field) and not re.match(r"^[0-9a-f]{32}$", row[field]):
            errors.append(field + " must be 32 hex characters")
    return errors


print(re.match(r"^\d{5}$", "14409") is not None)

True


In [7]:
# Task 7: Create custom exceptions: SchemaError and FileFormatError
class SchemaError(Exception):
    """A file has the wrong columns or too many bad rows."""


class FileFormatError(Exception):
    """A file is missing or cannot be read."""


print("Made SchemaError and FileFormatError")

Made SchemaError and FileFormatError


In [8]:
# Task 8: Handle a missing or empty file gracefully
def safe_read_csv(path):
    try:
        return read_rows(path)
    except FileNotFoundError:
        print("File not found:", path)
        return [], []


fields, rows = safe_read_csv(DATA_DIR / "no_file.csv")
print("Rows:", rows)

File not found: ..\archive\no_file.csv
Rows: []


In [9]:
# Task 9: Print a summary: total, valid and rejected rows
summary = {}
all_bad = []

for name, required in REQUIRED_FIELDS.items():
    fieldnames, rows = safe_read_csv(DATA_DIR / name)
    results = validate_every_row(rows, required)
    bad = [(row, errors) for row, errors in results if errors]
    all_bad += bad
    valid = len(results) - len(bad)
    summary[name] = [len(rows), valid, len(bad)]
    print(name, "total:", len(rows), "valid:", valid, "rejected:", len(bad))

write_bad_records(all_bad)
print("Summary:", summary)

olist_order_items_dataset.csv total: 112650 valid: 112650 rejected: 0


olist_orders_dataset.csv total: 99441 valid: 99441 rejected: 0


olist_customers_dataset.csv total: 99441 valid: 99441 rejected: 0
Wrote bad_records.csv
Summary: {'olist_order_items_dataset.csv': [112650, 112650, 0], 'olist_orders_dataset.csv': [99441, 99441, 0], 'olist_customers_dataset.csv': [99441, 99441, 0]}


In [10]:
# Task 10: Fail the whole file if more than 20% of rows are rejected (configurable)
import json
from pathlib import Path

if not Path("validation_config.json").exists():
    Path("validation_config.json").write_text('{"max_rejected_ratio": 0.20}')

with open("validation_config.json") as file:
    limit = json.load(file)["max_rejected_ratio"]

for name, counts in summary.items():
    total, valid, rejected = counts
    ratio = rejected / total
    print(name, round(ratio * 100, 1), "% rejected")
    if ratio > limit:
        raise SchemaError(name + " has too many rejected rows")

olist_order_items_dataset.csv 0.0 % rejected
olist_orders_dataset.csv 0.0 % rejected
olist_customers_dataset.csv 0.0 % rejected
